# Extract Part-wise Index ZIP Files

In [ ]:
import os, zipfile

ZIP_FILES = [
    "/content/index_folder1.zip",
    "/content/index_folder2.zip",
    "/content/index_folder3.zip",
    "/content/index_folder4.zip",
]

EXTRACT_BASE = "/content/index_parts_extracted"
os.makedirs(EXTRACT_BASE, exist_ok=True)

for i, z in enumerate(ZIP_FILES, start=1):
    out_dir = os.path.join(EXTRACT_BASE, f"part{i}")
    os.makedirs(out_dir, exist_ok=True)

    with zipfile.ZipFile(z, "r") as zip_ref:
        zip_ref.extractall(out_dir)

    print(f"✅ Extracted {z} -> {out_dir}")


✅ Extracted /content/index_folder1.zip -> /content/index_parts_extracted/part1
✅ Extracted /content/index_folder2.zip -> /content/index_parts_extracted/part2
✅ Extracted /content/index_folder3.zip -> /content/index_parts_extracted/part3
✅ Extracted /content/index_folder4.zip -> /content/index_parts_extracted/part4


In [ ]:
INDEX_FOLDERS = [
    "/content/index_parts_extracted/part1",
    "/content/index_parts_extracted/part2",
    "/content/index_parts_extracted/part3",
    "/content/index_parts_extracted/part4",
]


# Merge All Part-wise FAISS Indices into One Final Index
This cell loads the FAISS index and metadata from each part, then combines them into a single global search index.
It merges all embeddings into one FAISS file and unifies metadata dictionaries (paths, captions, attributes, region info).

In [ ]:
import os, pickle
import faiss

INDEX_FOLDERS = [
    "/content/index_parts_extracted/part1",
    "/content/index_parts_extracted/part2",
    "/content/index_parts_extracted/part3",
    "/content/index_parts_extracted/part4",
]

FINAL_INDEX_DIR = "/content/index_merged"
os.makedirs(FINAL_INDEX_DIR, exist_ok=True)

final_index = None
final_image_paths = []
final_captions = {}
final_attributes = {}
final_region_meta = {}

for i, folder in enumerate(INDEX_FOLDERS, start=1):
    print(f"\n✅ Loading extracted part {i}: {folder}")

    # ✅ load correct FAISS filename
    faiss_file = os.path.join(folder, f"faiss_index{i}.bin")
    if not os.path.exists(faiss_file):
        raise FileNotFoundError(f"❌ Missing: {faiss_file}")

    part_index = faiss.read_index(faiss_file)

    # ✅ Load metadata files (your naming also uses suffix)
    def load_any(folder, base_name):
        p2 = os.path.join(folder, f"{base_name}{i}.pkl")
        p1 = os.path.join(folder, f"{base_name}.pkl")
        if os.path.exists(p2): return p2
        if os.path.exists(p1): return p1
        raise FileNotFoundError(f"❌ Missing {base_name}{i}.pkl inside {folder}")

    image_paths_file = load_any(folder, "image_paths")
    captions_file    = load_any(folder, "captions")
    attributes_file  = load_any(folder, "attributes")
    region_file      = load_any(folder, "region_meta")

    with open(image_paths_file, "rb") as f:
        part_paths = pickle.load(f)
    with open(captions_file, "rb") as f:
        part_captions = pickle.load(f)
    with open(attributes_file, "rb") as f:
        part_attributes = pickle.load(f)
    with open(region_file, "rb") as f:
        part_region = pickle.load(f)

    # ✅ Initialize final index
    if final_index is None:
        final_index = faiss.IndexFlatIP(part_index.d)
        print("✅ Final index initialized. Dim:", part_index.d)

    # ✅ Add vectors
    vectors = part_index.reconstruct_n(0, part_index.ntotal)
    final_index.add(vectors)

    # ✅ Merge metadata
    final_image_paths.extend(part_paths)
    final_captions.update(part_captions)
    final_attributes.update(part_attributes)
    final_region_meta.update(part_region)

    print(f"✅ Added part {i} vectors:", part_index.ntotal)

# ✅ Save merged final index + metadata
faiss.write_index(final_index, os.path.join(FINAL_INDEX_DIR, "faiss_index.bin"))

with open(os.path.join(FINAL_INDEX_DIR, "image_paths.pkl"), "wb") as f:
    pickle.dump(final_image_paths, f)

with open(os.path.join(FINAL_INDEX_DIR, "captions.pkl"), "wb") as f:
    pickle.dump(final_captions, f)

with open(os.path.join(FINAL_INDEX_DIR, "attributes.pkl"), "wb") as f:
    pickle.dump(final_attributes, f)

with open(os.path.join(FINAL_INDEX_DIR, "region_meta.pkl"), "wb") as f:
    pickle.dump(final_region_meta, f)

print("\n MERGE COMPLETE")
print("Final total vectors:", final_index.ntotal)
print("Final total image paths:", len(final_image_paths))
print("Saved merged files to:", FINAL_INDEX_DIR)



✅ Loading extracted part 1: /content/index_parts_extracted/part1
✅ Final index initialized. Dim: 512
✅ Added part 1 vectors: 11243

✅ Loading extracted part 2: /content/index_parts_extracted/part2
✅ Added part 2 vectors: 11182

✅ Loading extracted part 3: /content/index_parts_extracted/part3
✅ Added part 3 vectors: 12040

✅ Loading extracted part 4: /content/index_parts_extracted/part4
✅ Added part 4 vectors: 11080

 MERGE COMPLETE
Final total vectors: 45545
Final total image paths: 45545
Saved merged files to: /content/index_merged
